In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [4]:
# read in data:
data = pd.read_excel('Clim_data.xlsx')
data

,Protein ID,Gene names,C-lim3_1,C-lim3_2,C-lim3_mean,C-lim6_1,C-lim6_2,C-lim6_mean,C-lim12_1,C-lim12_2,C-lim12_mean
0,sp|A5A614|YCIZ_ECOLI,yciZ,NaN,NaN,NaN,3.932642,3.601144,3.766893,NaN,NaN,NaN
1,sp|O32583|THIS_ECOLI,thiS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.469682,4.469682
2,sp|P00350|6PGD_ECOLI,gnd,3.248315,3.163151,3.205733,5.572427,6.131487,5.851957,10.201045,9.866244,10.033644
3,sp|P00363|FRDA_ECOLI,frdA,2.992966,2.946457,2.969711,6.080384,6.670409,6.375396,13.518765,10.761427,12.140096
4,sp|P00370|DHE4_ECOLI,gdhA,3.076857,2.810401,2.943629,4.462108,4.855534,4.658821,7.653326,6.91881,7.286068
...,...,...,...,...,...,...,...,...,...,...,...
2948,sp|Q46892|YGBN_ECOLI,ygbN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.22677,15.226770
2949,sp|Q46943|YQEJ_ECOLI,yqeJ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.000*,16.000000
2950,sp|Q47274|REQ1_ECOLI,quuD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.733813,8.733813
2951,sp|Q47534|YAIO_ECOLI,yaiO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.379927,15.379927


In [5]:
# Generate functions for cleaning the data

# Function for removing any rows that have NaN values in them: 
def remove_nan_rows(df):
    return df.dropna()

# Function for removing rows where an * appears somwhere in the row:
def remove_asterisk_rows(df):
    """
    Remove any rows that contain an asterisk (*) in any cell.
    The asterisk indicates that a ceiling value was applied to the data.
    Args:
        df: 

    Returns:

    """
    return df[~df.apply(lambda row: row.astype(str).str.contains('\*').any(), axis=1)]

# Function that removess the * symbol from any cell in the dataframe so that it is an actual number:
def remove_asterisk_symbols(df):
    """
    Remove asterisk (*) symbols from any cell in the dataframe.
    Args:
        df: pandas DataFrame

    Returns:
        pandas DataFrame with asterisks removed
    """
    return df.replace(to_replace=r'\*', value='', regex=True)

def convert_astrick_symbols_to_nan(df):
    """
    Convert asterisk (*) symbols to NaN in the dataframe.
    Args:
        df: pandas DataFrame

    Returns:
        pandas DataFrame with asterisks converted to NaN
    """
    return df.replace(to_replace=r'\*', value=np.nan, regex=True)

# Function that calculates the constant model active half life:
def calculate_constant_half_life(df, remove_negative_rows=False, remove_negative_values=False):
    """
    Calculate the constant model active half life from the data.
    Args:
        df: pandas DataFrame
        

    Returns:
        float, calculated half life
    """
    # only one of the optional parameters can be "on" at a time:
    if remove_negative_values == True and remove_negative_rows == True:
        raise ValueError("Cannot set both remove_negative_values and remove_negative_rows to True.")
    
        
    # name the columns: 
    mean_3 = df['C-lim3_mean']
    mean_6 = df['C-lim6_mean']
    mean_12 = df['C-lim12_mean']
    
    # implement the function: 
    df['HL_3vs6'] = 1/((mean_6/6 - mean_3/3) / (mean_3 - mean_6))
    
    df['HL_3vs12'] = 1/((mean_12/12 - mean_3/3) / (mean_3 - mean_12))
    
    df['HL_6vs12'] = 1/((mean_12/12 - mean_6/6) / (mean_6 - mean_12))
    
    # remove the negative values if needed (not their whole row):
    if remove_negative_values:
        df['HL_3vs6'] = df['HL_3vs6'].apply(lambda x: x if x > 0 else np.nan)
        df['HL_3vs12'] = df['HL_3vs12'].apply(lambda x: x if x > 0 else np.nan)
        df['HL_6vs12'] = df['HL_6vs12'].apply(lambda x: x if x > 0 else np.nan)
    
    # fully remove the rows with negative values if specified:
    if remove_negative_rows:
        df = df[(df['HL_3vs6'] > 0) & (df['HL_3vs12'] > 0) & (df['HL_6vs12'] > 0)].copy()
    
    df['constant_HL_model_mean'] = df[['HL_3vs6', 'HL_3vs12', 'HL_6vs12']].mean(axis=1).copy()
    df['constant_HL_model_STD'] = df[['HL_3vs6', 'HL_3vs12', 'HL_6vs12']].std(axis=1).copy()
    
    return df

# Define a function that calculates the standard half life using the ln(2)/k method:
def calculate_standard_half_life(df, remove_negative_rows=False, remove_negative_values=False):
    """
    Calculate the standard half life using the ln(2)/k method.
    Args:
        df: pandas DataFrame
            
                
    Returns:
        float, calculated half life
    """         
    # name the columns: 
    mean_3 = df['C-lim3_mean']
    mean_6 = df['C-lim6_mean']
    mean_12 = df['C-lim12_mean']
    
    # implement the function: #
    df['sHL_3'] = 1/((1/mean_3) - (1/3))
    df['sHL_6'] = 1/((1/mean_6) - (1/6))
    df['sHL_12'] = 1/((1/mean_12) - (1/12))
    
    # remove the negative values if needed (not their whole row):
    if remove_negative_values:
        df['sHL_3'] = df['sHL_3'].apply(lambda x: x if x > 0 else np.nan)
        df['sHL_6'] = df['sHL_6'].apply(lambda x: x if x > 0 else np.nan)
        df['sHL_12'] = df['sHL_12'].apply(lambda x: x if x > 0 else np.nan)   
        
    # fully remove the rows with negative values if specified:
    if remove_negative_rows:
        df = df[(df['sHL_3'] > 0) & (df['sHL_6'] > 0) & (df['sHL_12'] > 0)].copy()
    
    df['standard_HL_model_mean'] = df[['sHL_3', 'sHL_6', 'sHL_12']].mean(axis=1).copy()
    df['standard_HL_model_STD'] = df[['sHL_3', 'sHL_6', 'sHL_12']].std(axis=1).copy()
    
    return df
        
# Since the paper mentions that the 6 and 12 hour doubling time more accurately allow active half life to be measured, compute the metrics for just those replicates (but also calculate for the other two pairs): 
def calculate_constant_half_life_pair_metrics(df, remove_negative_rows=False, remove_negative_values=False):
    """
    Calculate the constant model active half life from the data using only 6 and 12 hour time points.
    Args:
        df: pandas DataFrame
        

    Returns:
        float, calculated half life
    """
    # only one of the optional parameters can be "on" at a time:
    if remove_negative_values == True and remove_negative_rows == True:
        raise ValueError("Cannot set both remove_negative_values and remove_negative_rows to True.")
    
    # check if the necessary columns exist:
    required_columns = ['HL_3vs6', 'HL_3vs12', 'HL_6vs12']
    if required_columns[0] not in df.columns:
        df = calculate_constant_half_life(df, remove_negative_rows=remove_negative_rows, remove_negative_values=remove_negative_values)
    
    # remove the negative values if needed (not their whole row):
    if remove_negative_values:
        df['HL_3vs6'] = df['HL_3vs6'].apply(lambda x: x if x > 0 else np.nan)
        df['HL_3vs12'] = df['HL_3vs12'].apply(lambda x: x if x > 0 else np.nan)
        df['HL_6vs12'] = df['HL_6vs12'].apply(lambda x: x if x > 0 else np.nan)
    
    # fully remove the rows with negative values if specified:
    if remove_negative_rows:
        df = df[(df['sHL_3'] > 0) & (df['sHL_6'] > 0) & (df['sHL_12'] > 0)].copy()
    
    df['constant_HL_model_3vs6_3vs12_mean'] = df[['HL_3vs6', 'HL_3vs12']].mean(axis=1).copy()
    df['constant_HL_model_3vs6_3vs12_STD'] = df[['HL_3vs6', 'HL_3vs12']].std(axis=1)
    df['constant_HL_model_3vs6_6vs12_mean'] = df[['HL_3vs6', 'HL_6vs12']].mean(axis=1).copy()
    df['constant_HL_model_3vs6_6vs12_STD'] = df[['HL_3vs6', 'HL_6vs12']].std(axis=1)
    df['constant_HL_model_3vs12_6vs12_mean'] = df[['HL_3vs12', 'HL_6vs12']].mean(axis=1).copy()
    df['constant_HL_model_3vs12_6vs12_STD'] = df[['HL_3vs12', 'HL_6vs12']].std(axis=1)
    
    # generate a column that indicates the smallest STD method for each row:
    std_columns = ['constant_HL_model_3vs6_3vs12_STD', 'constant_HL_model_3vs6_6vs12_STD', 'constant_HL_model_3vs12_6vs12_STD', 'constant_HL_model_STD']
    df['smallest_cHL_STD_method'] = df[std_columns].idxmin(axis=1)
    
    return df

# standard half life using only 6 and 12 hour time points:
def calculate_standard_half_life_pair_metrics(df, remove_negative_rows=False, remove_negative_values=False):
    """
    Calculate the standard half life using the ln(2)/k method using only 6 and 12 hour time points.
    Args:
        df: pandas DataFrame        
    Returns:
        float, calculated half life
    """
    # only one of the optional parameters can be "on" at a time:
    if remove_negative_values == True and remove_negative_rows == True:
        raise ValueError("Cannot set both remove_negative_values and remove_negative_rows to True.")
    
    # check if the necessary columns exist:
    required_columns = ['sHL_3', 'sHL_6', 'sHL_12']
    if required_columns[0] not in df.columns:
        df = calculate_standard_half_life(df, remove_negative_rows=remove_negative_rows, remove_negative_values=remove_negative_values)
    
    # remove the negative values if needed (not their whole row):
    if remove_negative_values:
        df['sHL_3'] = df['sHL_3'].apply(lambda x: x if x > 0 else np.nan)
        df['sHL_6'] = df['sHL_6'].apply(lambda x: x if x > 0 else np.nan)
        df['sHL_12'] = df['sHL_12'].apply(lambda x: x if x > 0 else np.nan)   
        
    # fully remove the rows with negative values if specified:
    if remove_negative_rows:
        df = df[(df['sHL_3'] > 0) & (df['sHL_6'] > 0) & (df['sHL_12'] > 0)].copy()
    
    df['standard_HL_model_3_6_mean'] = df[['sHL_3', 'sHL_6']].mean(axis=1).copy()
    df['standard_HL_model_3_6_STD'] = df[['sHL_3', 'sHL_6']].std(axis=1)
    df['standard_HL_model_3_12_mean'] = df[['sHL_3', 'sHL_12']].mean(axis=1).copy()
    df['standard_HL_model_3_12_STD'] = df[['sHL_3', 'sHL_12']].std(axis=1)
    df['standard_HL_model_6_12_mean'] = df[['sHL_6', 'sHL_12']].mean(axis=1).copy()
    df['standard_HL_model_6_12_STD'] = df[['sHL_6', 'sHL_12']].std(axis=1)
    
    # generate a column that indicates the smallest STD method for each row:
    std_columns = ['standard_HL_model_3_6_STD', 'standard_HL_model_3_12_STD', 'standard_HL_model_6_12_STD', 'standard_HL_model_STD']
    df['smallest_sHL_STD_method'] = df[std_columns].idxmin(axis=1)
    
    return df

# define a function that filters rows based on a standard deviation threshold:
def filter_by_std_threshold(df, method_column, std_threshold):
    """
    Filter rows based on a standard deviation threshold for a given method.
    Args:
        df: pandas DataFrame
        method_column: str, column name indicating the method used      
        std_threshold: float, standard deviation threshold
    Returns:
        pandas DataFrame, filtered DataFrame
    """
    # get the mean and std column names based on the method column:
    std_column = method_column
    # filter the dataframe:
    filtered_df = df[df[std_column] <= std_threshold]
    return filtered_df

# define a function that can cap half life values:
def cap_half_life_values(df, cap_value):
    """
    Cap half life values at a specified maximum value. 
    Args:
        df: pandas DataFrame
        cap_value: float, maximum half life value
    Returns:
        pandas DataFrame with capped half life values
    """
    # check that the HL_value column exists:
    if 'HL_value' not in df.columns:
        raise ValueError("HL_value column not found in DataFrame. Use standardized dataframe generation first.")
    
    # rename the HL_value column to HL_value_uncapped:
    df['HL_value_uncapped'] = df['HL_value']
    
    # iterate over each value in "HL_value_uncapped" and cap it if it exceeds the cap_value:
    df['HL_value'] = df['HL_value_uncapped'].apply(lambda x: min(x, cap_value))
    
    return df

# create a tsv file saving: 
def save_to_tsv(df, filename):
    file_name = '~/wcEcoli/out/clim_HL_sorts/' + filename + '.tsv'
    df.to_csv(file_name, sep='\t', index=False)

# TODO: add function that filters out by STD where it throws out any row who's ratio of HL to STD is below a certain threshold.
def filter_by_HL_to_STD_ratio(df, HL_column, STD_column, ratio_threshold, keep_NaNs=False):
    """
    Filter rows based on a half life to standard deviation ratio threshold.
    Args:
        df: pandas DataFrame
        HL_column: str, column name for half life values      
        STD_column: str, column name for standard deviation values
        ratio_threshold: float, half life to standard deviation ratio threshold
    Returns:
        pandas DataFrame, filtered DataFrame
    """
    # calculate the HL to STD ratio:
    df['HL_to_STD_ratio'] = df[HL_column] / df[STD_column]
    # filter the dataframe:
    if keep_NaNs:
        filtered_df = df[(df['HL_to_STD_ratio'] >= ratio_threshold) | (df['HL_to_STD_ratio'].isna())]
    else:
        filtered_df = df[df['HL_to_STD_ratio'] >= ratio_threshold]
    # drop the temporary ratio column:
    filtered_df = filtered_df.drop(columns=['HL_to_STD_ratio'])
    return filtered_df

 # NOW: it is made such that if you want to build something that searches if a negative value exists in a row for a particular value and if so, it can see if you can take the value from another column instead. If not, you can throw that row out entirely. ALSO implement an STD threshokd for filtering vaulues too 
# TODO: add a tag column that indicates which method was used to calculate the half life for each row.
# TODO: maybe choose the half life by the median value instead of the mean?
# TODO: compare STDs of the 3_6, 6_12, and 3_12 methods to see which is the most consistent? and choose an average of the smallest StD pair?


In [ ]:
# save just a large data frame of the data with all the calculated half lives:
clean_data = remove_asterisk_symbols(data)
clean_data = calculate_standard_half_life_pair_metrics(clean_data)
clean_data = calculate_constant_half_life_pair_metrics(clean_data)

file_name = '~/wcEcoli/out/clim_HL_sorts/large_dataframes/' + 'Clim_full_half_life_data.csv'
clean_data.to_csv(file_name, index=False)
clean_data

In [ ]:
# save just a large data frame of the data with all the calculated half lives:
def clean_data1(data):
    clean_data = remove_asterisk_symbols(data)
    clean_data = calculate_standard_half_life_pair_metrics(clean_data, remove_negative_values=True)
    clean_data = calculate_constant_half_life_pair_metrics(clean_data, remove_negative_values=True)
    # save to tsv: 
    file_name = '~/wcEcoli/out/clim_HL_sorts/large_dataframes/' + 'Clim_clean_data1.csv'
    clean_data.to_csv(file_name, index=False)
    print("clean_data1 output length:",len(clean_data))

def clean_data2(data):
    clean_data = convert_astrick_symbols_to_nan(data)
    clean_data = calculate_standard_half_life_pair_metrics(clean_data, remove_negative_values=True)
    clean_data = calculate_constant_half_life_pair_metrics(clean_data, remove_negative_values=True)
    # save to excel: 
    file_name = '~/wcEcoli/out/clim_HL_sorts/large_dataframes/' + 'Clim_clean_data2.csv'
    clean_data.to_csv(file_name, index=False)
    print("clean_data2 output length:",len(clean_data))

def clean_data3(data):
    clean_data = convert_astrick_symbols_to_nan(data)
    clean_data = remove_nan_rows(clean_data)
    clean_data = calculate_standard_half_life_pair_metrics(clean_data, remove_negative_values=True)
    clean_data = calculate_constant_half_life_pair_metrics(clean_data, remove_negative_values=True)
    # save to excel: 
    file_name = '~/wcEcoli/out/clim_HL_sorts/large_dataframes/' + 'Clim_clean_data3.csv'
    clean_data.to_csv(file_name, index=False)
    print("clean_data3 output length:",len(clean_data))

def clean_data4(data):
    clean_data = remove_asterisk_symbols(data)
    clean_data = calculate_standard_half_life_pair_metrics(clean_data, remove_negative_values=True)
    clean_data = calculate_constant_half_life_pair_metrics(clean_data, remove_negative_values=True)
    # save to excel: 
    file_name = '~/wcEcoli/out/clim_HL_sorts/large_dataframes/' + 'Clim_clean_data4.csv'
    clean_data.to_csv(file_name, index=False)
    print("clean_data4 output length:",len(clean_data))

# save just a large data frame of the data with all the calculated half lives:
def clean_data5(data):
    clean_data = convert_astrick_symbols_to_nan(data)
    clean_data = calculate_standard_half_life_pair_metrics(clean_data, remove_negative_values=True)
    clean_data = calculate_constant_half_life_pair_metrics(clean_data, remove_negative_values=True)
    # save to excel: 
    file_name = '~/wcEcoli/out/clim_HL_sorts/large_dataframes/' + 'Clim_clean_data5.csv'
    clean_data.to_csv(file_name, index=False)
    print("clean_data5 output length:",len(clean_data))

# save just a large data frame of the data with all the calculated half lives:
def clean_data6(data):
    clean_data = convert_astrick_symbols_to_nan(data)
    clean_data = remove_nan_rows(clean_data)
    clean_data = calculate_standard_half_life_pair_metrics(clean_data, remove_negative_values=True)
    clean_data = calculate_constant_half_life_pair_metrics(clean_data, remove_negative_values=True)
    # save to excel: 
    file_name = '~/wcEcoli/out/clim_HL_sorts/large_dataframes/' + 'Clim_clean_data6.csv'
    clean_data.to_csv(file_name, index=False)
    print("clean_data6 output length:",len(clean_data))
    
clean_data6(data)

def clean_data7(data):
    clean_data = convert_astrick_symbols_to_nan(data)
    clean_data = remove_nan_rows(clean_data)
    clean_data = calculate_standard_half_life_pair_metrics(clean_data, remove_negative_rows=True)
    clean_data = calculate_constant_half_life_pair_metrics(clean_data, remove_negative_rows=True)
    # save to a csv: 
    file_name = '~/wcEcoli/out/clim_HL_sorts/large_dataframes/' + 'Clim_clean_data7.csv'
    clean_data.to_csv(file_name, index=False)
    print("clean_data7 output length:",len(clean_data))
    
clean_data7(data)

def clean_data8(data):
    clean_data = remove_asterisk_symbols(data)
    clean_data = remove_nan_rows(clean_data)
    clean_data = calculate_standard_half_life_pair_metrics(clean_data, remove_negative_values=True)
    clean_data = calculate_constant_half_life_pair_metrics(clean_data, remove_negative_values=True)
    # save to a csv: 
    file_name = '~/wcEcoli/out/clim_HL_sorts/large_dataframes/' + 'Clim_clean_data8.csv'
    clean_data.to_csv(file_name, index=False)
    print("clean_data8 output length:",len(clean_data))
    
clean_data8(data)


clean_data8 output length: 2022


In [205]:
def generate_df4(data, STD_threshold=None, cap_HL_value=None, STD_ratio_threshold=None, keep_NaNs_in_ratio_filter=False):
    # convert * symbols to NaN:
    df = convert_astrick_symbols_to_nan(data)
    # calculate half lives while removing any rows with negative HL values:
    df = calculate_standard_half_life_pair_metrics(df, remove_negative_values=True)
    df = calculate_constant_half_life_pair_metrics(df, remove_negative_values=True)
    
    # Generate a dataframe with the desired columns only:
    HL_selections = df[['Protein ID','Gene names ']].copy()    
    
    # Create a new column for HL_value and fill it accordingly:
    HL_selections['selected_HL'] = None
    HL_selections['HL_value'] = None
    HL_selections['HL_value_STD'] = None
    for index, row in df.iterrows():
        sHL_STD_min = row['smallest_sHL_STD_method']
        cHL_STD_min = row['smallest_cHL_STD_method']
       
        # initialize HL candidates:
        cHL_candidate = None
        cHL_candidate_value = None
        sHL_candidate = None
        sHL_candidate_value = None
        
        # define what the final value will be: 
        hl_selection = None
        hl_value = None
        hl_STD = None
        cHL_to_sHL_ratio = None
        cHL_STD_to_sHL_STD_ratio = None
        
       
        # Must check if either of these are NaN first:
        if pd.isna(cHL_STD_min):
            # Check if the constant_HL_model_mean column exists:
            total_mean_value = df.at[index, 'constant_HL_model_mean']
            if np.isnan(total_mean_value) == True:
                # If not, assign NaN to the HL_value and HL_value_STD columns so that it can be removed later:
                cHL_candidate = np.nan
                
            else:
                # Assign to the constant_HL_model_mean column instead:
                cHL_candidate_temp = 'constant_HL_model_mean'
                cHL_candidate_value_temp = df.at[index, cHL_candidate_temp]
                # check if the 'constant_HL_model_mean' value matches any of the values of HL_3vs6, HL_3vs12, or HL_6vs12 values and just name it that if so:
                if np.isclose(cHL_candidate_value_temp, df.at[index, 'HL_3vs6'], equal_nan=False):
                    cHL_candidate = 'HL_3vs6'
                    #cHL_candidate_value = df.at[index, cHL_candidate]
                elif np.isclose(cHL_candidate_value_temp, df.at[index, 'HL_3vs12'], equal_nan=False):
                    cHL_candidate = 'HL_3vs12'
                    #cHL_candidate_value = df.at[index, cHL_candidate]
                elif np.isclose(cHL_candidate_value_temp, df.at[index, 'HL_6vs12'], equal_nan=False):
                    cHL_candidate = 'HL_6vs12'
                    #cHL_candidate_value = df.at[index, cHL_candidate]
                else:
                    cHL_candidate = 'constant_HL_model_mean'
                
                cHL_candidate_value = df.at[index, cHL_candidate]
                
        else:
            # if the original value is not NaN, assign it to the candidate:
            cHL_candidate = cHL_STD_min.replace('STD', 'mean')
            cHL_candidate_value = df.at[index, cHL_candidate]
            
                
        if pd.isna(sHL_STD_min):
            # Check if the standard_HL_model_mean column exists:
            total_mean_value = df.at[index, 'standard_HL_model_mean']
            if np.isnan(total_mean_value) == True:
                # If not, assign NaN to the HL_value and HL_value_STD columns so that it can be removed later:
                sHL_candidate = np.nan
            else:
                # Assign to the standard_HL_model_mean column instead:
                sHL_candidate_temp = 'standard_HL_model_mean'
                sHL_candidate_value_temp = df.at[index, sHL_candidate_temp]
                # check if the 'standard_HL_model_mean' value matches any of the values of sHL_3, sHL_6, or sHL_12 values and just name it that if so:
                if np.isclose(sHL_candidate_value_temp, df.at[index, 'sHL_3'], equal_nan=False):
                    sHL_candidate = 'sHL_3'
                    sHL_candidate_value = df.at[index, sHL_candidate]
                elif np.isclose(sHL_candidate_value_temp, df.at[index, 'sHL_6'], equal_nan=False):
                    sHL_candidate = 'sHL_6'
                    sHL_candidate_value = df.at[index, sHL_candidate]
                elif np.isclose(sHL_candidate_value_temp, df.at[index, 'sHL_12'], equal_nan=False):
                    sHL_candidate = 'sHL_12'
                    sHL_candidate_value = df.at[index, sHL_candidate]
                else:
                    sHL_candidate = 'standard_HL_model_mean'
                    sHL_candidate_value = df.at[index, sHL_candidate]
                
        else:
            # if the original value is not NaN, assign it to the candidate:
            sHL_candidate = sHL_STD_min.replace('STD', 'mean')
            sHL_candidate_value = df.at[index, sHL_candidate]
            
        # Now, determine best value to use based on the candidate values:
        if pd.isna(sHL_candidate) and pd.isna(cHL_candidate):
            hl_value = np.nan
            hl_STD = np.nan
            cHL_to_sHL_ratio = np.nan
            # later this row will be removed!
            
        elif pd.isna(sHL_candidate) and not pd.isna(cHL_candidate):
            hl_value = cHL_candidate_value
            if not pd.isna(cHL_STD_min):
                hl_STD = df.at[index, cHL_STD_min]
            else:
                hl_STD = np.nan
            hl_selection = cHL_candidate
            cHL_to_sHL_ratio = np.nan
            
        elif pd.isna(cHL_candidate) and not pd.isna(sHL_candidate):
            hl_value = sHL_candidate_value
            if not pd.isna(sHL_STD_min):
                hl_STD = df.at[index, sHL_STD_min]
            else:
                hl_STD = np.nan
            hl_selection = sHL_candidate
            cHL_to_sHL_ratio = np.nan
        
        else:
            # both candidates are valid, choose the one with the smaller STD:
            
            # first check if the STD values are NaN. if so, arbrirtrarily choose the smaller HL value:
            if pd.isna(sHL_STD_min) and pd.isna(cHL_STD_min):
                hl_value = min(sHL_candidate_value, cHL_candidate_value)
                hl_STD = np.nan
                hl_selection = sHL_candidate if sHL_candidate_value <= cHL_candidate_value else cHL_candidate # todo: decide if this is ok to do.... maybe just assign to 12
                cHL_to_sHL_ratio = np.nan

            # if the standard HL STD is NaN but the constant HL STD is not, choose the constant HL:
            elif pd.isna(sHL_STD_min) and not pd.isna(cHL_STD_min):
                hl_value = cHL_candidate_value
                hl_STD = df.at[index, cHL_STD_min]
                hl_selection = cHL_candidate
                cHL_to_sHL_ratio = np.nan
            
            # if the constant HL STD is NaN but the standard HL STD is not, choose the standard HL:            
            elif pd.isna(cHL_STD_min) and not pd.isna(sHL_STD_min):
                hl_value = sHL_candidate_value
                hl_STD = df.at[index, sHL_STD_min]
                hl_selection = sHL_candidate
                cHL_to_sHL_ratio = np.nan
                
                
            # both STD values are valid, choose the one with the smaller STD:
            else: 
                sHL_STD_value = df.at[index, sHL_STD_min]
                cHL_STD_value = df.at[index, cHL_STD_min]
                if sHL_STD_value <= cHL_STD_value:
                    hl_value = sHL_candidate_value
                    hl_STD = sHL_STD_value
                    hl_selection = sHL_candidate
                else:
                    hl_value = cHL_candidate_value
                    hl_STD = cHL_STD_value
                    hl_selection = cHL_candidate
                # calculate ratio:
                cHL_to_sHL_ratio = cHL_candidate_value / sHL_candidate_value 
                cHL_STD_to_sHL_STD_ratio = cHL_STD_value / sHL_STD_value 
                
        # assign the values to the dataframe:
        HL_selections.at[index, 'selected_HL'] = hl_selection
        HL_selections.at[index, 'HL_value'] = hl_value
        HL_selections.at[index, 'HL_value_STD'] = hl_STD
        HL_selections.at[index, 'cHL_to_sHL_ratio'] = cHL_to_sHL_ratio
        HL_selections.at[index, 'cHL_STD_to_sHL_STD_ratio'] = cHL_STD_to_sHL_STD_ratio
        HL_selections.at[index, 'sHL_candidate'] = sHL_candidate
        HL_selections.at[index, 'sHL_candidate_value'] = sHL_candidate_value
        HL_selections.at[index, 'cHL_candidate'] = cHL_candidate
        HL_selections.at[index, 'cHL_candidate_value'] = cHL_candidate_value
        
    # Now, remove any rows where selected_HL is NaN:
    HL_selections = HL_selections[HL_selections['selected_HL'].notna()]
    
                    
    if STD_threshold is not None:
        HL_selections = filter_by_std_threshold(HL_selections, 'HL_value_STD', STD_threshold)
        
    if STD_ratio_threshold is not None:
        HL_selections = filter_by_HL_to_STD_ratio(HL_selections, 'HL_value', 'HL_value_STD', STD_ratio_threshold, keep_NaNs=keep_NaNs_in_ratio_filter)
        
    if cap_HL_value is not None:
        HL_selections = cap_half_life_values(HL_selections, cap_HL_value)
        
    
    
    df4 = HL_selections  
    return df4

df4 = generate_df4(data, STD_ratio_threshold=2, keep_NaNs_in_ratio_filter=True)
# save to a tsv file:
save_to_tsv(df4, 'Clim4_STD_ratio_threshold_2_keep_NaNs')

df4



,Protein ID,Gene names,selected_HL,HL_value,HL_value_STD,cHL_to_sHL_ratio,cHL_STD_to_sHL_STD_ratio,sHL_candidate,sHL_candidate_value,cHL_candidate,cHL_candidate_value
0,sp|A5A614|YCIZ_ECOLI,yciZ,sHL_6,10.121036,NaN,NaN,NaN,sHL_6,10.121036,NaN,NaN
1,sp|O32583|THIS_ECOLI,thiS,sHL_12,7.122697,NaN,NaN,NaN,sHL_12,7.122697,NaN,NaN
2,sp|P00350|6PGD_ECOLI,gnd,constant_HL_model_3vs12_6vs12_mean,29.709034,0.472581,0.199119,0.003799,standard_HL_model_6_12_mean,149.202179,constant_HL_model_3vs12_6vs12_mean,29.709034
3,sp|P00363|FRDA_ECOLI,frdA,HL_6vs12,113.274499,NaN,NaN,NaN,sHL_3,294.141533,HL_6vs12,113.274499
4,sp|P00370|DHE4_ECOLI,gdhA,standard_HL_model_6_12_mean,19.694901,1.62232,0.507417,1.408795,standard_HL_model_6_12_mean,19.694901,constant_HL_model_3vs6_3vs12_mean,9.993536
...,...,...,...,...,...,...,...,...,...,...,...
2943,sp|P65298|YQHH_ECOLI,yqhH,sHL_12,14.351835,NaN,NaN,NaN,sHL_12,14.351835,NaN,NaN
2945,sp|P76073|YNAE_ECOLI,ynaE,sHL_12,8.352574,NaN,NaN,NaN,sHL_12,8.352574,NaN,NaN
2946,sp|P76458|ATOD_ECOLI,atoD,sHL_12,29.663415,NaN,NaN,NaN,sHL_12,29.663415,NaN,NaN
2947,sp|P77162|YKFB_ECOLI,ykfB,sHL_12,9.589682,NaN,NaN,NaN,sHL_12,9.589682,NaN,NaN


In [92]:
# make a dataframe that is very conservative and pulls from the standard half life values: 
def generate_df5(data, STD_threshold=None, cap_HL_value=None):
    # remove all rows with any NaN values:
    df = remove_nan_rows(data)
    # remove rows with * symbols:
    df = remove_asterisk_rows(df)
    # calculate half lives while removing any rows with negative HL values:
    df = calculate_standard_half_life_pair_metrics(df, remove_negative_rows=True)
    df = calculate_constant_half_life_pair_metrics(df, remove_negative_rows=True)
    
    # Generate a dataframe with the desired columns only:
    HL_selections = df[['Protein ID','Gene names ']].copy()
    
    # record selected half life method based on smallest STD from standard half life method:
    HL_selections['selected_HL'] = df['smallest_sHL_STD_method'].str.replace('_STD', '_mean')

    # Create a new column for HL_value and fill it accordingly:
    HL_selections['HL_value'] = None
    HL_selections['HL_value_STD'] = None
    for index, row in HL_selections.iterrows():
        hl_column = row['selected_HL']
        std_colum = hl_column.replace('mean', 'STD')
        if hl_column in df.columns:
            HL_selections.at[index, 'HL_value'] = df.at[index, hl_column]
            HL_selections.at[index, 'HL_value_STD'] = df.at[index, std_colum]
    
    if STD_threshold is not None:
        HL_selections = filter_by_std_threshold(HL_selections, 'HL_value_STD', STD_threshold)
        
    if cap_HL_value is not None:
        HL_selections = cap_half_life_values(HL_selections, cap_HL_value)
    
    df5 = HL_selections  
    return df5

df5 = generate_df5(data)
# save to a tsv file:
save_to_tsv(df5, 'Clim5')

# save a version with a STD threshold of 2 hours:
df5_std_cap2 = generate_df5(data, STD_threshold=2)
save_to_tsv(df5_std_cap2, 'Clim5_STD_cap_2')

# save a version with a HL cap of 12 hours:
df5_hl_cap12 = generate_df5(data, cap_HL_value=12)
save_to_tsv(df5_hl_cap12, 'Clim5_HL_cap_12')

df5_hl_cap12


,Protein ID,Gene names,selected_HL,HL_value,HL_value_STD,HL_value_uncapped
4,sp|P00370|DHE4_ECOLI,gdhA,standard_HL_model_6_12_mean,12.000000,1.62232,19.694901
14,sp|P00579|RPOD_ECOLI,rpoD,standard_HL_model_6_12_mean,12.000000,2.104101,25.336875
20,sp|P00861|DCDA_ECOLI,lysA,standard_HL_model_6_12_mean,12.000000,1.197414,23.940561
21,sp|P00864|CAPP_ECOLI,ppc,standard_HL_model_6_12_mean,12.000000,20.49379,143.899155
23,sp|P00888|AROF_ECOLI,aroF,standard_HL_model_6_12_mean,5.420453,0.315323,5.420453
...,...,...,...,...,...,...
2706,sp|P77743|PRPR_ECOLI,prpR,standard_HL_model_6_12_mean,11.884631,0.423738,11.884631
2711,sp|Q46799|XDHA_ECOLI,xdhA,standard_HL_model_3_12_mean,5.031422,0.043832,5.031422
2712,sp|Q46801|XDHC_ECOLI,xdhC,standard_HL_model_3_6_mean,3.286340,0.606829,3.28634
2713,sp|Q46803|YGEW_ECOLI,ygeW,standard_HL_model_6_12_mean,12.000000,9.044516,32.757093


In [117]:
# make an analouge to df5 but with the constant half life method:
def generate_df6(data, STD_threshold=None, cap_HL_value=None):
    # remove all rows with any NaN values:
    df = remove_nan_rows(data)
    # remove rows with * symbols:
    df = remove_asterisk_rows(df)
    # calculate half lives while removing any rows with negative HL values:
    df = calculate_standard_half_life_pair_metrics(df, remove_negative_rows=True)
    df = calculate_constant_half_life_pair_metrics(df, remove_negative_rows=True)
    
    # Generate a dataframe with the desired columns only:
    HL_selections = df[['Protein ID','Gene names ']].copy()
    
    # record selected half life method based on smallest STD from standard half life method:
    HL_selections['selected_HL'] = df['smallest_cHL_STD_method'].str.replace('_STD', '_mean')

    # Create a new column for HL_value and fill it accordingly:
    HL_selections['HL_value'] = None
    HL_selections['HL_value_STD'] = None
    for index, row in HL_selections.iterrows():
        hl_column = row['selected_HL']
        std_column = hl_column.replace('mean', 'STD')
        if hl_column in df.columns:
            HL_selections.at[index, 'HL_value'] = df.at[index, hl_column]
            HL_selections.at[index, 'HL_value_STD'] = df.at[index, std_column]
    
    if STD_threshold is not None:
        HL_selections = filter_by_std_threshold(HL_selections, 'HL_value_STD', STD_threshold)
        
    if cap_HL_value is not None:
        HL_selections = cap_half_life_values(HL_selections, cap_HL_value)
    
    df6 = HL_selections  
    return df6

df6 = generate_df6(data)
# save to a tsv file:
save_to_tsv(df6, 'Clim6')

# save a version with a STD threshold of 2 hours:
df6_std_cap2 = generate_df6(data, STD_threshold=2)
save_to_tsv(df6_std_cap2, 'Clim6_STD_cap_2')

# save a version with a HL cap of 12 hours:
df6_hl_cap12 = generate_df6(data, cap_HL_value=12)
save_to_tsv(df6_hl_cap12, 'Clim6_HL_cap_12')

df6_hl_cap12


,Protein ID,Gene names,selected_HL,HL_value,HL_value_STD,HL_value_uncapped
4,sp|P00370|DHE4_ECOLI,gdhA,constant_HL_model_3vs6_3vs12_mean,9.993536,2.285516,9.993536
14,sp|P00579|RPOD_ECOLI,rpoD,constant_HL_model_3vs12_6vs12_mean,12.000000,4.355485,23.349217
20,sp|P00861|DCDA_ECOLI,lysA,constant_HL_model_3vs6_3vs12_mean,12.000000,4.926786,13.926156
21,sp|P00864|CAPP_ECOLI,ppc,constant_HL_model_3vs12_6vs12_mean,12.000000,0.932248,106.875892
23,sp|P00888|AROF_ECOLI,aroF,constant_HL_model_3vs6_3vs12_mean,2.826053,1.284622,2.826053
...,...,...,...,...,...,...
2706,sp|P77743|PRPR_ECOLI,prpR,constant_HL_model_3vs12_6vs12_mean,12.000000,3.771946,13.146572
2711,sp|Q46799|XDHA_ECOLI,xdhA,constant_HL_model_3vs12_6vs12_mean,4.734465,0.563917,4.734465
2712,sp|Q46801|XDHC_ECOLI,xdhC,constant_HL_model_3vs6_3vs12_mean,9.119026,1.743463,9.119026
2713,sp|Q46803|YGEW_ECOLI,ygeW,constant_HL_model_3vs6_3vs12_mean,12.000000,12.191289,20.269065


In [159]:
# make a less conservative dataframe that pulls from the constant half life values, even if there are Nan values in the data:
def generate_df8(data, STD_threshold=None, cap_HL_value=None):
    # convert * symbols to NaN:
    df = convert_astrick_symbols_to_nan(data)
    # calculate half lives while removing any rows with negative HL values:
    df = calculate_standard_half_life_pair_metrics(df, remove_negative_values=True)
    df = calculate_constant_half_life_pair_metrics(df, remove_negative_values=True)
    
    # Generate a dataframe with the desired columns only:
    HL_selections = df[['Protein ID','Gene names ']].copy()
    
    # record selected half life method based on smallest STD from constant half life method:
    HL_selections['selected_HL'] = df['smallest_cHL_STD_method']
    
    # Create a new column for HL_value and fill it accordingly:
    HL_selections['HL_value'] = None
    HL_selections['HL_value_STD'] = None
    for index, row in HL_selections.iterrows():
        hl_column = row['selected_HL']
        # Check what the value of the the selected HL column is:
        if pd.isna(hl_column):
            # Check if the constant_HL_model_mean column exists:
            total_mean_value = df.at[index, 'constant_HL_model_mean']
            if np.isnan(total_mean_value) == True:
                # If not, assign NaN to the HL_value and HL_value_STD columns so that it can be removed later:
                HL_selections.at[index, 'HL_value'] = np.nan
                HL_selections.at[index, 'HL_value_STD'] = np.nan
            else:
                # Assign to the constant_HL_model_mean column instead:
                hl_column = 'constant_HL_model_mean'
                HL_selections.at[index, 'selected_HL'] = hl_column
                HL_selections.at[index, 'HL_value'] = df.at[index, hl_column]
                HL_selections.at[index, 'HL_value_STD'] = np.nan 
            
        if hl_column in df.columns:
            hl_column = hl_column.replace('STD', 'mean')
            HL_selections.at[index, 'selected_HL'] = hl_column
            std_column = hl_column.replace('mean', 'STD')
            HL_selections.at[index, 'HL_value'] = df.at[index, hl_column]
            HL_selections.at[index, 'HL_value_STD'] = df.at[index, std_column]
            
    # remove any rows where selected_HL is NaN:
    HL_selections = HL_selections[HL_selections['selected_HL'].notna()]
    
    if STD_threshold is not None:
        HL_selections = filter_by_std_threshold(HL_selections, 'HL_value_STD', STD_threshold)
        
    if cap_HL_value is not None:
        HL_selections = cap_half_life_values(HL_selections, cap_HL_value)
    
    df8 = HL_selections  
    return df8

df8 = generate_df8(data)
# save to a tsv file:
save_to_tsv(df8, 'Clim8')

df8_std_cap2 = generate_df8(data, STD_threshold=2)
save_to_tsv(df8_std_cap2, 'Clim8_STD_cap_2')
df8_hl_cap12 = generate_df8(data, cap_HL_value=12)
save_to_tsv(df8_hl_cap12, 'Clim8_HL_cap_12')
df8_std_cap2



,Protein ID,Gene names,selected_HL,HL_value,HL_value_STD
2,sp|P00350|6PGD_ECOLI,gnd,constant_HL_model_3vs12_6vs12_mean,29.709034,0.472581
21,sp|P00864|CAPP_ECOLI,ppc,constant_HL_model_3vs12_6vs12_mean,106.875892,0.932248
23,sp|P00888|AROF_ECOLI,aroF,constant_HL_model_3vs6_3vs12_mean,2.826053,1.284622
29,sp|P00909|TRPC_ECOLI,trpC,constant_HL_model_3vs12_6vs12_mean,14.731321,0.124962
34,sp|P00936|CYAA_ECOLI,cyaA,constant_HL_model_3vs6_3vs12_mean,4.953403,1.006209
...,...,...,...,...,...
2777,sp|P60932|UPPP_ECOLI,uppP,constant_HL_model_3vs12_6vs12_mean,23.128291,0.372
2782,sp|P75733|CHIP_ECOLI,chiP,constant_HL_model_3vs12_6vs12_mean,2.554603,0.858055
2796,sp|P76461|ATOB_ECOLI,atoB,constant_HL_model_3vs12_6vs12_mean,10.504327,0.072953
2804,sp|P77579|PTFC1_ECOLI,fryC,constant_HL_model_3vs12_6vs12_mean,8.823194,1.505146


In [158]:
# make a less conservative dataframe that pulls from the standard half life values, even if there are Nan values in the data:
def generate_df7(data, STD_threshold=None, cap_HL_value=None):
    # convert * symbols to NaN:
    df = convert_astrick_symbols_to_nan(data)
    # calculate half lives while removing any rows with negative HL values:
    df = calculate_standard_half_life_pair_metrics(df, remove_negative_values=True)
    df = calculate_constant_half_life_pair_metrics(df, remove_negative_values=True)
    
    # Generate a dataframe with the desired columns only:
    HL_selections = df[['Protein ID','Gene names ']].copy()
    
    # record selected half life method based on smallest STD from standard half life method:
    HL_selections['selected_HL'] = df['smallest_sHL_STD_method']
    
    # Create a new column for HL_value and fill it accordingly:
    HL_selections['HL_value'] = None
    HL_selections['HL_value_STD'] = None
    for index, row in HL_selections.iterrows():
        hl_column = row['selected_HL']
        # Check what the value of the the selected HL column is:
        if pd.isna(hl_column):
            # Check if the standard_HL_model_mean column exists:
            total_mean_value = df.at[index, 'standard_HL_model_mean']
            if np.isnan(total_mean_value) == True:
                # If not, assign NaN to the HL_value and HL_value_STD columns so that it can be removed later:
                HL_selections.at[index, 'HL_value'] = np.nan
                HL_selections.at[index, 'HL_value_STD'] = np.nan
            else:
                # Assign to the standard_HL_model_mean column instead:
                hl_column = 'standard_HL_model_mean'
                HL_selections.at[index, 'selected_HL'] = hl_column
                HL_selections.at[index, 'HL_value'] = df.at[index, hl_column]
                HL_selections.at[index, 'HL_value_STD'] = np.nan 
            
        if hl_column in df.columns:
            hl_column = hl_column.replace('STD', 'mean')
            HL_selections.at[index, 'selected_HL'] = hl_column
            std_column = hl_column.replace('mean', 'STD')
            HL_selections.at[index, 'HL_value'] = df.at[index, hl_column]
            HL_selections.at[index, 'HL_value_STD'] = df.at[index, std_column]
            
    # remove any rows where selected_HL is NaN:
    HL_selections = HL_selections[HL_selections['selected_HL'].notna()]
    
    if STD_threshold is not None:
        HL_selections = filter_by_std_threshold(HL_selections, 'HL_value_STD', STD_threshold)
        
    if cap_HL_value is not None:
        HL_selections = cap_half_life_values(HL_selections, cap_HL_value)
    
    df7 = HL_selections  
    return df7

df7 = generate_df7(data)


,Protein ID,Gene names,selected_HL,HL_value,HL_value_STD
0,sp|A5A614|YCIZ_ECOLI,yciZ,standard_HL_model_mean,10.121036,NaN
1,sp|O32583|THIS_ECOLI,thiS,standard_HL_model_mean,7.122697,NaN
2,sp|P00350|6PGD_ECOLI,gnd,standard_HL_model_6_12_mean,149.202179,124.408746
3,sp|P00363|FRDA_ECOLI,frdA,standard_HL_model_mean,294.141533,NaN
4,sp|P00370|DHE4_ECOLI,gdhA,standard_HL_model_6_12_mean,19.694901,1.62232
...,...,...,...,...,...
2943,sp|P65298|YQHH_ECOLI,yqhH,standard_HL_model_mean,14.351835,NaN
2945,sp|P76073|YNAE_ECOLI,ynaE,standard_HL_model_mean,8.352574,NaN
2946,sp|P76458|ATOD_ECOLI,atoD,standard_HL_model_mean,29.663415,NaN
2947,sp|P77162|YKFB_ECOLI,ykfB,standard_HL_model_mean,9.589682,NaN


In [160]:
def generate_df9(data, STD_threshold=None, cap_HL_value=None):
    # convert * symbols to NaN:
    df = convert_astrick_symbols_to_nan(data)
    # remove rows with NaN values:
    df = remove_nan_rows(df)
    # calculate half lives while removing any rows with negative HL values:
    df = calculate_standard_half_life_pair_metrics(df, remove_negative_values=True)
    df = calculate_constant_half_life_pair_metrics(df, remove_negative_values=True)
    
    # Generate a dataframe with the desired columns only:
    HL_selections = df[['Protein ID','Gene names ']].copy()
    
    # record selected half life method based on smallest STD from constant half life method:
    HL_selections['selected_HL'] = df['smallest_cHL_STD_method']
    
    # Create a new column for HL_value and fill it accordingly:
    HL_selections['HL_value'] = None
    HL_selections['HL_value_STD'] = None
    for index, row in HL_selections.iterrows():
        hl_column = row['selected_HL']
        # Check what the value of the the selected HL column is:
        if pd.isna(hl_column):
            # Check if the constant_HL_model_mean column exists:
            total_mean_value = df.at[index, 'constant_HL_model_mean']
            if np.isnan(total_mean_value) == True:
                # If not, assign NaN to the HL_value and HL_value_STD columns so that it can be removed later:
                HL_selections.at[index, 'HL_value'] = np.nan
                HL_selections.at[index, 'HL_value_STD'] = np.nan
            else:
                # Assign to the constant_HL_model_mean column instead:
                hl_column = 'constant_HL_model_mean'
                HL_selections.at[index, 'selected_HL'] = hl_column
                HL_selections.at[index, 'HL_value'] = df.at[index, hl_column]
                HL_selections.at[index, 'HL_value_STD'] = np.nan 
            
        if hl_column in df.columns:
            hl_column = hl_column.replace('STD', 'mean')
            HL_selections.at[index, 'selected_HL'] = hl_column
            std_column = hl_column.replace('mean', 'STD')
            HL_selections.at[index, 'HL_value'] = df.at[index, hl_column]
            HL_selections.at[index, 'HL_value_STD'] = df.at[index, std_column]
            
    # remove any rows where selected_HL is NaN:
    HL_selections = HL_selections[HL_selections['selected_HL'].notna()]
    
    if STD_threshold is not None:
        HL_selections = filter_by_std_threshold(HL_selections, 'HL_value_STD', STD_threshold)
        
    if cap_HL_value is not None:
        HL_selections = cap_half_life_values(HL_selections, cap_HL_value)
    
    df9 = HL_selections  
    return df9

df9 = generate_df9(data)
# save to a tsv file:
save_to_tsv(df9, 'Clim9')
df9



,Protein ID,Gene names,selected_HL,HL_value,HL_value_STD
2,sp|P00350|6PGD_ECOLI,gnd,constant_HL_model_3vs12_6vs12_mean,29.709034,0.472581
3,sp|P00363|FRDA_ECOLI,frdA,constant_HL_model_mean,113.274499,NaN
4,sp|P00370|DHE4_ECOLI,gdhA,constant_HL_model_3vs6_3vs12_mean,9.993536,2.285516
7,sp|P00452|RIR1_ECOLI,nrdA,constant_HL_model_3vs6_3vs12_mean,16.734578,10.839045
9,sp|P00509|AAT_ECOLI,aspC,constant_HL_model_3vs6_3vs12_mean,27.420952,12.468417
...,...,...,...,...,...
2712,sp|Q46801|XDHC_ECOLI,xdhC,constant_HL_model_3vs6_3vs12_mean,9.119026,1.743463
2713,sp|Q46803|YGEW_ECOLI,ygeW,constant_HL_model_3vs6_3vs12_mean,20.269065,12.191289
2714,sp|Q46822|IDI_ECOLI,idi,constant_HL_model_3vs12_6vs12_mean,16.084441,6.471766
2715,sp|Q46891|OTNI_ECOLI,otnI,constant_HL_model_3vs6_3vs12_mean,5.958238,4.764496


In [193]:
def generate_df11(data, STD_threshold=None, cap_HL_value=None, STD_ratio_threshold=None, keep_NaNs_in_ratio_filter=False):
    # remove * symbols:
    df = remove_asterisk_symbols(data)
    # remove rows with NaN values:
    df = remove_nan_rows(df)    
    # calculate half lives while removing any rows with negative HL values:
    df = calculate_standard_half_life_pair_metrics(df, remove_negative_values=True)
    df = calculate_constant_half_life_pair_metrics(df, remove_negative_values=True)
    
    # Generate a dataframe with the desired columns only:
    HL_selections = df[['Protein ID','Gene names ']].copy()    
    
    # Create a new column for HL_value and fill it accordingly:
    HL_selections['selected_HL'] = None
    HL_selections['HL_value'] = None
    HL_selections['HL_value_STD'] = None
    for index, row in df.iterrows():
        sHL_STD_min = row['smallest_sHL_STD_method']
        cHL_STD_min = row['smallest_cHL_STD_method']
       
        # initialize HL candidates:
        cHL_candidate = None
        cHL_candidate_value = None
        sHL_candidate = None
        sHL_candidate_value = None
        
        # define what the final value will be: 
        hl_selection = None
        hl_value = None
        hl_STD = None
        cHL_to_sHL_ratio = None
        cHL_STD_to_sHL_STD_ratio = None
        
       
        # Must check if either of these are NaN first:
        if pd.isna(cHL_STD_min):
            # Check if the constant_HL_model_mean column exists:
            total_mean_value = df.at[index, 'constant_HL_model_mean']
            if np.isnan(total_mean_value) == True:
                # If not, assign NaN to the HL_value and HL_value_STD columns so that it can be removed later:
                cHL_candidate = np.nan
                
            else:
                # Assign to the constant_HL_model_mean column instead:
                cHL_candidate = 'constant_HL_model_mean'
                cHL_candidate_value = df.at[index, cHL_candidate]
                
        else:
            # if the original value is not NaN, assign it to the candidate:
            cHL_candidate = cHL_STD_min.replace('STD', 'mean')
            cHL_candidate_value = df.at[index, cHL_candidate]
            
                
        if pd.isna(sHL_STD_min):
            # Check if the standard_HL_model_mean column exists:
            total_mean_value = df.at[index, 'standard_HL_model_mean']
            if np.isnan(total_mean_value) == True:
                # If not, assign NaN to the HL_value and HL_value_STD columns so that it can be removed later:
                sHL_candidate = np.nan
            else:
                # Assign to the standard_HL_model_mean column instead:
                sHL_candidate = 'standard_HL_model_mean'
                sHL_candidate_value = df.at[index, sHL_candidate]
                
        else:
            # if the original value is not NaN, assign it to the candidate:
            sHL_candidate = sHL_STD_min.replace('STD', 'mean')
            sHL_candidate_value = df.at[index, sHL_candidate]
            
        # Now, determine best value to use based on the candidate values:
        if pd.isna(sHL_candidate) and pd.isna(cHL_candidate):
            hl_value = np.nan
            hl_STD = np.nan
            cHL_to_sHL_ratio = np.nan
            # later this row will be removed!
            
        elif pd.isna(sHL_candidate) and not pd.isna(cHL_candidate):
            hl_value = cHL_candidate_value
            if not pd.isna(cHL_STD_min):
                hl_STD = df.at[index, cHL_STD_min]
            else:
                hl_STD = np.nan
            hl_selection = cHL_candidate
            cHL_to_sHL_ratio = np.nan
            
        elif pd.isna(cHL_candidate) and not pd.isna(sHL_candidate):
            hl_value = sHL_candidate_value
            if not pd.isna(sHL_STD_min):
                hl_STD = df.at[index, sHL_STD_min]
            else:
                hl_STD = np.nan
            hl_selection = sHL_candidate
            cHL_to_sHL_ratio = np.nan
        
        else:
            # both candidates are valid, choose the one with the smaller STD:
            
            # first check if the STD values are NaN. if so, arbrirtrarily choose the smaller HL value:
            if pd.isna(sHL_STD_min) and pd.isna(cHL_STD_min):
                hl_value = min(sHL_candidate_value, cHL_candidate_value)
                hl_STD = np.nan
                hl_selection = sHL_candidate if sHL_candidate_value <= cHL_candidate_value else cHL_candidate
                cHL_to_sHL_ratio = np.nan

            # if the standard HL STD is NaN but the constant HL STD is not, choose the constant HL:
            elif pd.isna(sHL_STD_min) and not pd.isna(cHL_STD_min):
                hl_value = cHL_candidate_value
                hl_STD = df.at[index, cHL_STD_min]
                hl_selection = cHL_candidate
                cHL_to_sHL_ratio = np.nan
            
            # if the constant HL STD is NaN but the standard HL STD is not, choose the standard HL:            
            elif pd.isna(cHL_STD_min) and not pd.isna(sHL_STD_min):
                hl_value = sHL_candidate_value
                hl_STD = df.at[index, sHL_STD_min]
                hl_selection = sHL_candidate
                cHL_to_sHL_ratio = np.nan
                
                
            # both STD values are valid, choose the one with the smaller STD:
            else: 
                sHL_STD_value = df.at[index, sHL_STD_min]
                cHL_STD_value = df.at[index, cHL_STD_min]
                if sHL_STD_value <= cHL_STD_value:
                    hl_value = sHL_candidate_value
                    hl_STD = sHL_STD_value
                    hl_selection = sHL_candidate
                else:
                    hl_value = cHL_candidate_value
                    hl_STD = cHL_STD_value
                    hl_selection = cHL_candidate
                # calculate ratio:
                cHL_to_sHL_ratio = cHL_candidate_value / sHL_candidate_value 
                cHL_STD_to_sHL_STD_ratio = cHL_STD_value / sHL_STD_value 
                
        # assign the values to the dataframe:
        HL_selections.at[index, 'selected_HL'] = hl_selection
        HL_selections.at[index, 'HL_value'] = hl_value
        HL_selections.at[index, 'HL_value_STD'] = hl_STD
        HL_selections.at[index, 'cHL_to_sHL_ratio'] = cHL_to_sHL_ratio
        HL_selections.at[index, 'cHL_STD_to_sHL_STD_ratio'] = cHL_STD_to_sHL_STD_ratio
        
    # Now, remove any rows where selected_HL is NaN:
    HL_selections = HL_selections[HL_selections['selected_HL'].notna()]
    
    
                    
    if STD_threshold is not None:
        HL_selections = filter_by_std_threshold(HL_selections, 'HL_value_STD', STD_threshold)
        
    if STD_ratio_threshold is not None:
        HL_selections = filter_by_HL_to_STD_ratio(HL_selections, 'HL_value', 'HL_value_STD', STD_ratio_threshold, keep_NaNs=keep_NaNs_in_ratio_filter)
        
    if cap_HL_value is not None:
        HL_selections = cap_half_life_values(HL_selections, cap_HL_value)
        
    
    
    df11 = HL_selections  
    return df11

df11b = generate_df11(data, cap_HL_value=12, STD_ratio_threshold=2, keep_NaNs_in_ratio_filter=True)
# save to a tsv file:
save_to_tsv(df11b, 'Clim11_STD_ratio_threshold_2_keep_NaNs_HL_cap_12')

df11b



,Protein ID,Gene names,selected_HL,HL_value,HL_value_STD,cHL_to_sHL_ratio,cHL_STD_to_sHL_STD_ratio,HL_value_uncapped
2,sp|P00350|6PGD_ECOLI,gnd,constant_HL_model_3vs12_6vs12_mean,12.000000,0.472581,0.199119,0.003799,29.709034
3,sp|P00363|FRDA_ECOLI,frdA,constant_HL_model_mean,12.000000,NaN,NaN,NaN,113.274499
4,sp|P00370|DHE4_ECOLI,gdhA,standard_HL_model_6_12_mean,12.000000,1.62232,0.507417,1.408795,19.694901
7,sp|P00452|RIR1_ECOLI,nrdA,standard_HL_model_6_12_mean,12.000000,9.165655,0.497545,1.182572,33.634327
9,sp|P00509|AAT_ECOLI,aspC,standard_HL_model_6_12_mean,12.000000,1.821761,0.372636,6.844157,73.586516
...,...,...,...,...,...,...,...,...
2711,sp|Q46799|XDHA_ECOLI,xdhA,standard_HL_model_3_12_mean,5.031422,0.043832,0.940979,12.865281,5.031422
2712,sp|Q46801|XDHC_ECOLI,xdhC,standard_HL_model_3_6_mean,3.286340,0.606829,2.774827,2.873072,3.28634
2713,sp|Q46803|YGEW_ECOLI,ygeW,standard_HL_model_6_12_mean,12.000000,9.044516,0.618769,1.347921,32.757093
2714,sp|Q46822|IDI_ECOLI,idi,constant_HL_model_3vs12_6vs12_mean,12.000000,6.471766,NaN,NaN,16.084441


In [195]:
def generate_df12(data, STD_threshold=None, cap_HL_value=None, STD_ratio_threshold=None, keep_NaNs_in_ratio_filter=False):
    # remove * symbols:
    df = convert_astrick_symbols_to_nan(data)
        
    # calculate half lives while removing any rows with negative HL values:
    df = calculate_standard_half_life_pair_metrics(df, remove_negative_values=True)
    df = calculate_constant_half_life_pair_metrics(df, remove_negative_values=True)
    
    # Generate a dataframe with the desired columns only:
    HL_selections = df[['Protein ID','Gene names ']].copy()    
    
    # Create a new column for HL_value and fill it accordingly:
    HL_selections['selected_HL'] = None
    HL_selections['HL_value'] = None
    HL_selections['HL_value_STD'] = None
    for index, row in df.iterrows():
        sHL_12_hour = row['sHL_12']
        cHL_6vs12_hour = row['HL_6vs12']
               
        # define what the final value will be: 
        hl_selection = None
        hl_value = None
        hl_STD = None
        cHL_to_sHL_ratio = None
        cHL_STD_to_sHL_STD_ratio = None
        
        if not pd.isna(sHL_12_hour):
            hl_selection = 'sHL_12'
            hl_value = df.at[index, hl_selection]
        elif pd.isna(sHL_12_hour) and not pd.isna(cHL_6vs12_hour):
            hl_selection = 'HL_6vs12'
            hl_value = df.at[index, hl_selection]
        else:
            hl_value = np.nan
            hl_selection = np.nan
       
        
        # assign the values to the dataframe:
        HL_selections.at[index, 'selected_HL'] = hl_selection
        HL_selections.at[index, 'HL_value'] = hl_value
        
    # Now, remove any rows where selected_HL is NaN:
    HL_selections = HL_selections[HL_selections['selected_HL'].notna()]
    
    
                    
    if STD_threshold is not None:
        HL_selections = filter_by_std_threshold(HL_selections, 'HL_value_STD', STD_threshold)
        
    if STD_ratio_threshold is not None:
        HL_selections = filter_by_HL_to_STD_ratio(HL_selections, 'HL_value', 'HL_value_STD', STD_ratio_threshold, keep_NaNs=keep_NaNs_in_ratio_filter)
        
    if cap_HL_value is not None:
        HL_selections = cap_half_life_values(HL_selections, cap_HL_value)
        
    
    
    df12 = HL_selections  
    return df12

df12b = generate_df12(data)
# save to a tsv file:
save_to_tsv(df12b, 'Clim12')
df12b


,Protein ID,Gene names,selected_HL,HL_value,HL_value_STD
1,sp|O32583|THIS_ECOLI,thiS,sHL_12,7.122697,None
2,sp|P00350|6PGD_ECOLI,gnd,sHL_12,61.231911,None
3,sp|P00363|FRDA_ECOLI,frdA,HL_6vs12,113.274499,None
4,sp|P00370|DHE4_ECOLI,gdhA,sHL_12,18.547748,None
5,sp|P00393|DHNA_ECOLI,ndh,sHL_12,645.921188,None
...,...,...,...,...,...
2943,sp|P65298|YQHH_ECOLI,yqhH,sHL_12,14.351835,None
2945,sp|P76073|YNAE_ECOLI,ynaE,sHL_12,8.352574,None
2946,sp|P76458|ATOD_ECOLI,atoD,sHL_12,29.663415,None
2947,sp|P77162|YKFB_ECOLI,ykfB,sHL_12,9.589682,None
